<a href="https://colab.research.google.com/github/jyotidabass/Customizing-LLM-Architectures-Techniques-for-modifying-model-architectures-for-specific-use-cases/blob/main/Customizing_LLM_Architectures_Techniques_for_modifying_model_architectures_for_specific_use%C2%A0cases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers

import torch
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

texts = [
    "I love this movie!",
    "I hate this movie!",
    "This movie is okay.",
    "I'm not sure about this movie.",
    "This movie is amazing!",
]
labels = [1, 0, 1, 1, 1]

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
max_len = 64  # Reduced max_len to avoid potential memory issues

dataset = SentimentDataset(texts, labels, tokenizer, max_len)
data_loader = DataLoader(dataset, batch_size=16, shuffle=True)

class SentimentModel(nn.Module):
    def __init__(self):
        super(SentimentModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(self.bert.config.hidden_size, 2)  # 2 output neurons for binary classification

    def forward(self, input_ids, attention_mask):
        # Get the pooled output from BERT, ensuring return_dict=False for older versions of transformers
        _, pooled_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=False)
        output = self.drop(pooled_output)
        return self.out(output)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SentimentModel()
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

for epoch in range(3):  # Reduced epochs for faster execution
    model.train()
    total_loss = 0
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(data_loader)}')

model.eval()
total_correct = 0
with torch.no_grad():
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)
        total_correct += (predicted == labels).sum().item()

accuracy = total_correct / len(labels)
print(f'Accuracy: {accuracy:.4f}')

Epoch 1, Loss: 0.6570695638656616
Epoch 2, Loss: 0.7910135984420776
Epoch 3, Loss: 0.5908066034317017
Accuracy: 0.8000
